In [0]:
customers = spark.table("retail_silver.customers")

orders = spark.table("retail_silver.orders")

order_items = spark.table("retail_silver.order_items")

products = spark.table("retail_silver.products")

categories = spark.table("retail_silver.categories")

departments = spark.table("retail_silver.departments")

In [0]:
customers.createOrReplaceTempView('customers')
orders.createOrReplaceTempView('orders')
order_items.createOrReplaceTempView('order_items')
products.createOrReplaceTempView('products')
categories.createOrReplaceTempView('categories')
departments.createOrReplaceTempView('departments')



In [0]:
%sql
SELECT order_status,
count(*) as total_count
FROM orders
GROUP BY order_status
ORDER BY total_count DESC

order_status,total_count
COMPLETE,22899
PENDING_PAYMENT,15030
PROCESSING,8275
PENDING,7610
CLOSED,7556
ON_HOLD,3798
SUSPECTED_FRAUD,1558
CANCELED,1428
PAYMENT_REVIEW,729


In [0]:
%sql
SELECT round(sum(order_item_subtotal),2) as Toatal_revenue
FROM order_items


Toatal_revenue
3.432261993E7


In [0]:
%sql
SELECT p.product_name,
       round(sum(oi.order_item_subtotal),2) as Total_revenue
FROM order_items oi
JOIN products p 
ON oi.order_item_product_id = p.product_id
GROUP BY product_name
ORDER BY Total_revenue DESC   
LIMIT 10    

product_name,Total_revenue
Field & Stream Sportsman 16 Gun Fire Safe,6929653.5
Perfect Fitness Perfect Rip Deck,4421143.02
Diamondback Women's Serene Classic Comfort Bi,4118425.42
Nike Men's Free 5.0+ Running Shoe,3667633.2
Nike Men's Dri-FIT Victory Golf Polo,3147800.0
Pelican Sunstream 100 Kayak,3099845.0
Nike Men's CJ Elite 2 TD Football Cleat,2891757.54
O'Brien Men's Neoprene Life Vest,2888993.94
Under Armour Girls' Toddler Spine Surge Runni,1269082.65
adidas Youth Germany Black/Red Away Match Soc,67830.0


In [0]:
%sql
SELECT
    c.category_name,
    ROUND(SUM(oi.order_item_subtotal),2) AS total_revenue
FROM order_items oi
JOIN products p
ON oi.order_item_product_id = p.product_id
JOIN categories c
ON p.product_cateogry_id = c.category_id
GROUP BY c.category_name
ORDER BY total_revenue DESC

category_name,total_revenue
Fishing,6929653.5
Cleats,4431942.66
Camping & Hiking,4118425.42
Cardio Equipment,3694843.2
Women's Apparel,3147800.0
Water Sports,3113844.6
Men's Footwear,2891757.54
Indoor/Outdoor Games,2888993.94
Shop By Sport,1309522.02
Electronics,371034.64


In [0]:
%sql
SELECT
    c.category_name,
    ROUND(SUM(oi.order_item_subtotal),2) AS total_revenue
FROM order_items oi
JOIN products p
ON oi.order_item_product_id = p.product_id
JOIN categories c
ON p.product_cateogry_id = c.category_id
GROUP BY c.category_name
ORDER BY total_revenue DESC;

category_name,total_revenue
Fishing,6929653.5
Cleats,4431942.66
Camping & Hiking,4118425.42
Cardio Equipment,3694843.2
Women's Apparel,3147800.0
Water Sports,3113844.6
Men's Footwear,2891757.54
Indoor/Outdoor Games,2888993.94
Shop By Sport,1309522.02
Electronics,371034.64


In [0]:
%sql

SELECT 
    customer_state,
    COUNT(*) AS total_customers
FROM customers
GROUP BY customer_state
ORDER BY total_customers DESC
LIMIT 10

customer_state,total_customers
PR,4771
CA,2012
NY,775
TX,635
IL,523
FL,374
OH,276
PA,261
MI,254
NJ,219


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS retail_gold;
use retail_gold;
SHOW TABLES


database,tableName,isTemporary
retail_gold,category_revenue,false
retail_gold,customer_state_analysis,false
retail_gold,order_status_analysis,false
retail_gold,revenue_kpi,false
retail_gold,top_products,false
,categories,true
,customers,true
,departments,true
,order_items,true
,orders,true


In [0]:
%sql
CREATE OR REPLACE TABLE retail_gold.revenue_kpi AS

SELECT
    ROUND(SUM(order_item_subtotal),2) AS total_revenue,
    COUNT(DISTINCT order_item_order_id) AS total_orders,
    COUNT(DISTINCT order_item_product_id) AS total_products
FROM retail_silver.order_items;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE retail_gold.category_revenue AS

SELECT
    c.category_name,
    ROUND(SUM(oi.order_item_subtotal),2) AS total_revenue
FROM retail_silver.order_items oi
JOIN retail_silver.products p
ON oi.order_item_product_id = p.product_id
JOIN retail_silver.categories c
ON p.product_cateogry_id = c.category_id
GROUP BY c.category_name
ORDER BY total_revenue DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE retail_gold.order_status_analysis AS

SELECT 
    order_status,
    COUNT(*) AS total_orders
FROM retail_silver.orders
GROUP BY order_status
ORDER BY total_orders DESC

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE retail_gold.customer_state_analysis AS

SELECT
    customer_state,
    COUNT(*) AS total_customers
FROM retail_silver.customers
GROUP BY customer_state
ORDER BY total_customers DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE retail_gold.top_products AS

WITH product_revenue AS (

SELECT
    p.product_name,
    ROUND(SUM(oi.order_item_subtotal),2) AS revenue
FROM retail_silver.order_items oi
JOIN retail_silver.products p
ON oi.order_item_product_id = p.product_id
GROUP BY p.product_name

)

SELECT *
FROM product_revenue
ORDER BY revenue DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SHOW TABLES IN retail_gold;

database,tableName,isTemporary
retail_gold,category_revenue,false
retail_gold,customer_state_analysis,false
retail_gold,order_status_analysis,false
retail_gold,revenue_kpi,false
retail_gold,top_products,false
,categories,true
,customers,true
,departments,true
,order_items,true
,orders,true
